#  ⭐ `fat_reacoes`


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root, normalize_date_column, normalize_rows, apply_mappings
project_root = get_project_root() 

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Reacoes/Reacoes.parquet"
bronze = pd.read_parquet(path) 
bronze.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT', 'PT',
       'HLT', 'HLGT', 'SOC', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA', 'DURACAO',
       'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

In [4]:
bronze.head(2)

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT,PT,HLT,HLGT,SOC,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17


### GRAVE

In [5]:
bronze.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

### DESFECHO

In [6]:
bronze.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

### GRAVIDADE

In [7]:
bronze.GRAVIDADE.value_counts()

GRAVIDADE
Outro efeito clinicamente significativo                                                                                                                                                                              144833
Hospitalização/Prolongamento de hospitalização                                                                                                                                                                        57343
Hospitalização/Prolongamento de hospitalização, Outro efeito clinicamente significativo                                                                                                                               15621
Resultou em óbito                                                                                                                                                                                                     11510
Ameaça à vida                                                                                                 

In [8]:
 
gravidades_unicas = (
    bronze['GRAVIDADE']
    .dropna()
    .astype(str)
    .str.split(r'\s*,\s*')   # separa pelos ","
    .explode()               # uma linha por item
    .str.strip()             # tira espaços extras
    .drop_duplicates()       # remove duplicados
    .sort_values()           # opcional: ordena
)

print(gravidades_unicas.to_list())


['Ameaça à vida', 'Anomalia congênita ou malformação ao nascer', 'Hospitalização/Prolongamento de hospitalização', 'Incapacidade persistente ou significativa', 'Outro efeito clinicamente significativo', 'Resultou em óbito']


In [9]:
for g in gravidades_unicas:
    print(g)

Ameaça à vida
Anomalia congênita ou malformação ao nascer
Hospitalização/Prolongamento de hospitalização
Incapacidade persistente ou significativa
Outro efeito clinicamente significativo
Resultou em óbito


### DURACAO

In [10]:
bronze.DURACAO.value_counts().head(10)

DURACAO
1 dia        27059
2 dia        10017
3 dia         6650
1 hora        6314
30 minuto     5865
4 dia         4278
5 dia         3707
0 dia         3349
2 hora        3112
20 minuto     2698
Name: count, dtype: int64

# 🥈 Silver

normalized data, medium quality


In [11]:
silver= bronze.rename(columns={"REACAO_EVTO_ADVERSO_MEDDRA_LLT": "REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR","SOC":"SOC_VALOR"})
silver.head()

,IDENTIFICACAO_NOTIFICACAO,REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR,PT,HLT,HLGT,SOC_VALOR,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO
0,BR-ANVISA-300000004,Coceira,Prurido,Prurido NCO,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17
1,BR-ANVISA-300000005,Edema periorbital,Edema periorbital,Distúrbios oculares NCO,Transtornos oculares NCO,Distúrbios oculares,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
2,BR-ANVISA-300000007,Exantema alérgico,Dermatite alérgica,Dermatite e eczema,Quadros clínicos epidérmicos e dérmicos,Distúrbios dos tecidos cutâneos e subcutâneos,20181115,None,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
3,BR-ANVISA-300000008,Flebite,Flebite,Flebite NCO,Infecções e inflamações vasculares,Distúrbios vasculares,20181025,None,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17
4,BR-ANVISA-300000010,Parestesia,Parestesia,Parestesias e disestesias,Distúrbios neurológicos NCO,Distúrbios do sistema nervoso,201508,201508,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17


## ✅ hash

In [12]:
silver.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR',
       'PT', 'HLT', 'HLGT', 'SOC_VALOR', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'GRAVE', 'GRAVIDADE', 'DESFECHO', 'ATUALIZADO'],
      dtype='object')

In [13]:
from utils import build_row_hash

hash_cols = ['IDENTIFICACAO_NOTIFICACAO', 'REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR',
       'PT', 'HLT', 'HLGT', 'SOC_VALOR', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'GRAVE', 'GRAVIDADE', 'DESFECHO']  # as colunas que devem entrar no hash
silver["HASH_BRONZE"] = build_row_hash(silver, hash_cols, algo="sha256", sep="|")

## ✅missing

In [14]:
silver.isna().sum()

IDENTIFICACAO_NOTIFICACAO                    0
REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR     16288
PT                                       16288
HLT                                      16288
HLGT                                     16288
SOC_VALOR                                16288
DATA_INICIO_HORA                        359162
DATA_FINAL_HORA                         583697
DURACAO                                 688151
GRAVE                                   173081
GRAVIDADE                               548073
DESFECHO                                135672
ATUALIZADO                                   0
HASH_BRONZE                                  0
dtype: int64

## ✅ chave_soc  chave_llt

In [15]:
dim_soc = pd.read_parquet(Path(project_root) / "data/03_gold/dim_soc_llt/dim_soc_llt.parquet")
dim_soc.head()

,SOC_CHAVE,SOC_VALOR,HLGT,HLT,PT,LLT_CHAVE,REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR
0,26,Circunstâncias sociais,Fatores relacionados ao gênero,Circunstâncias relacionadas à gravidez,Amamentação,1,Amamentação
1,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Abstinência sexual,2,Abstinência sexual
2,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Atividade sexual aumentada,3,Atividade sexual aumentada
3,26,Circunstâncias sociais,Fatores relacionados ao gênero,Questões de sexualidade,Não consumação,4,Não consumação
4,26,Circunstâncias sociais,Fatores relacionados à idade,Questões relacionadas à idade,Bebê,5,Neonato


In [16]:
dim_soc.columns

Index(['SOC_CHAVE', 'SOC_VALOR', 'HLGT', 'HLT', 'PT', 'LLT_CHAVE',
       'REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR'],
      dtype='object')

In [17]:
hist_fat_reacoes = silver.merge(dim_soc, on=['REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR', 'PT', 'HLT', 'HLGT','SOC_VALOR'], how='left')
hist_fat_reacoes = hist_fat_reacoes.drop(['SOC_VALOR', 'HLGT', 'HLT', 'PT','REACAO_EVTO_ADVERSO_MEDDRA_LLT_VALOR'], axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO,HASH_BRONZE,SOC_CHAVE,LLT_CHAVE
0,BR-ANVISA-300000004,None,None,3 dia,Não,None,Recuperado/Resolvido,2025-11-17,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,16,4114
1,BR-ANVISA-300000005,20181122,20181122,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,9,9107
2,BR-ANVISA-300000007,20181115,None,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,16,3903
3,BR-ANVISA-300000008,20181025,None,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,12,11411
4,BR-ANVISA-300000010,201508,201508,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,8,1994


## ✅DATAS

In [18]:
col_dates = ["DATA_INICIO_HORA", "DATA_FINAL_HORA"]

for col in col_dates:
    if col in hist_fat_reacoes.columns:
        hist_fat_reacoes[col] = normalize_date_column(hist_fat_reacoes[col])

hist_fat_reacoes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815877 entries, 0 to 815876
Data columns (total 11 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   IDENTIFICACAO_NOTIFICACAO  815877 non-null  object        
 1   DATA_INICIO_HORA           347696 non-null  datetime64[ns]
 2   DATA_FINAL_HORA            189475 non-null  datetime64[ns]
 3   DURACAO                    127726 non-null  object        
 4   GRAVE                      642796 non-null  object        
 5   GRAVIDADE                  267804 non-null  object        
 6   DESFECHO                   680205 non-null  object        
 7   ATUALIZADO                 815877 non-null  object        
 8   HASH_BRONZE                815877 non-null  object        
 9   SOC_CHAVE                  770393 non-null  Int64         
 10  LLT_CHAVE                  815877 non-null  int64         
dtypes: Int64(1), datetime64[ns](2), int64(1), object(7)


In [19]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVE,GRAVIDADE,DESFECHO,ATUALIZADO,HASH_BRONZE,SOC_CHAVE,LLT_CHAVE
0,BR-ANVISA-300000004,NaT,NaT,3 dia,Não,None,Recuperado/Resolvido,2025-11-17,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,16,4114
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,9,9107
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,16,3903
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Sim,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,12,11411
4,BR-ANVISA-300000010,NaT,NaT,None,Sim,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,8,1994


## ✅ GRAVE

In [20]:
hist_fat_reacoes.GRAVE.value_counts()

GRAVE
Não    374682
Sim    268114
Name: count, dtype: int64

In [21]:
 
grave_map = {
    "Desconhecido": 0,
    "Sim": 1,
    "Não": 2, 
}
hist_fat_reacoes['GRAVE_VALOR'] = hist_fat_reacoes['GRAVE'].fillna("Desconhecido") 
hist_fat_reacoes['GRAVE_CHAVE'] = hist_fat_reacoes['GRAVE'].map(grave_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes.drop('GRAVE', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,DESFECHO,ATUALIZADO,HASH_BRONZE,SOC_CHAVE,LLT_CHAVE,GRAVE_VALOR,GRAVE_CHAVE
0,BR-ANVISA-300000004,NaT,NaT,3 dia,None,Recuperado/Resolvido,2025-11-17,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,16,4114,Não,2
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,9,9107,Sim,1
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,16,3903,Sim,1
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,Recuperado/Resolvido,2025-11-17,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,12,11411,Sim,1
4,BR-ANVISA-300000010,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,Recuperado/Resolvido,2025-11-17,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,8,1994,Sim,1


In [22]:
hist_fat_reacoes[['GRAVE_CHAVE', 'GRAVE_VALOR']].drop_duplicates()

,GRAVE_CHAVE,GRAVE_VALOR
0,2,Não
1,1,Sim
36,0,Desconhecido


## ✅ DESFECHO

In [23]:
hist_fat_reacoes.DESFECHO.value_counts()

DESFECHO
Recuperado/Resolvido                         278670
Desconhecido                                 226494
Não Recuperado/Não Resolvido/Em andamento     85241
Em recuperação/Resolvendo                     66908
Fatal/Óbito                                   18581
Recuperado/Resolvido com sequelas              4311
Name: count, dtype: int64

In [24]:
desfecho_map = {
    "Desconhecido": 0, 
    "Não Recuperado/Não Resolvido/Em andamento": 1, 
    "Em recuperação/Resolvendo": 2, 
    "Recuperado/Resolvido": 3,
    "Fatal/Óbito": 4, 
    "Recuperado/Resolvido com sequelas": 5, 
}
hist_fat_reacoes['DESFECHO_VALOR'] = hist_fat_reacoes['DESFECHO'].fillna("Desconhecido")  
hist_fat_reacoes['DESFECHO_CHAVE'] = hist_fat_reacoes['DESFECHO'].map(desfecho_map).astype('Int64').fillna(0)
hist_fat_reacoes = hist_fat_reacoes.drop('DESFECHO', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,HASH_BRONZE,SOC_CHAVE,LLT_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE
0,BR-ANVISA-300000004,NaT,NaT,3 dia,None,2025-11-17,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,16,4114,Não,2,Recuperado/Resolvido,3
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,2025-11-17,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,9,9107,Sim,1,Recuperado/Resolvido,3
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2025-11-17,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,16,3903,Sim,1,Recuperado/Resolvido,3
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2025-11-17,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,12,11411,Sim,1,Recuperado/Resolvido,3
4,BR-ANVISA-300000010,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,2025-11-17,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,8,1994,Sim,1,Recuperado/Resolvido,3


In [25]:
hist_fat_reacoes[['DESFECHO_CHAVE', 'DESFECHO_VALOR']].drop_duplicates()

,DESFECHO_CHAVE,DESFECHO_VALOR
0,3,Recuperado/Resolvido
5,0,Desconhecido
39,2,Em recuperação/Resolvendo
53,1,Não Recuperado/Não Resolvido/Em andamento
64,4,Fatal/Óbito
331,5,Recuperado/Resolvido com sequelas


## ✅ DURACAO

In [26]:
from utils import normalize_duracao

In [27]:
hist_fat_reacoes.DURACAO.value_counts().head(10)

DURACAO
1 dia        27059
2 dia        10017
3 dia         6650
1 hora        6314
30 minuto     5865
4 dia         4278
5 dia         3707
0 dia         3349
2 hora        3112
20 minuto     2698
Name: count, dtype: int64

In [28]:
hist_fat_reacoes = normalize_duracao(hist_fat_reacoes, coluna="DURACAO")

In [29]:
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,HASH_BRONZE,SOC_CHAVE,LLT_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_VALOR
0,BR-ANVISA-300000004,NaT,NaT,3 dia,None,2025-11-17,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,16,4114,Não,2,Recuperado/Resolvido,3,DIA,4,3.0
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,2025-11-17,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,9,9107,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2025-11-17,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,16,3903,Sim,1,Recuperado/Resolvido,3,DIA,4,2.0
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2025-11-17,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,12,11411,Sim,1,Recuperado/Resolvido,3,DIA,4,5.0
4,BR-ANVISA-300000010,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,2025-11-17,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,8,1994,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN


In [30]:
hist_fat_reacoes.DURACAO_VALOR.value_counts().head(15)

DURACAO_VALOR
1.0     37118
2.0     15475
3.0      9916
30.0     6419
5.0      6188
4.0      6144
10.0     4269
6.0      3669
0.0      3479
20.0     3369
7.0      3269
15.0     2735
8.0      2652
40.0     1978
12.0     1612
Name: count, dtype: int64

### GRAVIDADE

In [31]:
from utils import expandir_gravidade_wide

In [32]:
hist_fat_reacoes = expandir_gravidade_wide(hist_fat_reacoes, col='GRAVIDADE', prefix='GRAVIDADE_')
#hist_fat_reacoes = hist_fat_reacoes.drop('GRAVIDADE', axis=1)
hist_fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,DATA_INICIO_HORA,DATA_FINAL_HORA,DURACAO,GRAVIDADE,ATUALIZADO,HASH_BRONZE,SOC_CHAVE,LLT_CHAVE,GRAVE_VALOR,GRAVE_CHAVE,DESFECHO_VALOR,DESFECHO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_TIPO_CHAVE,DURACAO_VALOR,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA
0,BR-ANVISA-300000004,NaT,NaT,3 dia,None,2025-11-17,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,16,4114,Não,2,Recuperado/Resolvido,3,DIA,4,3.0,0,0,0,0,0,0
1,BR-ANVISA-300000005,2018-11-22,2018-11-22,None,Outro efeito clinicamente significativo,2025-11-17,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,9,9107,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN,0,0,0,0,1,0
2,BR-ANVISA-300000007,2018-11-15,NaT,2 dia,Outro efeito clinicamente significativo,2025-11-17,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,16,3903,Sim,1,Recuperado/Resolvido,3,DIA,4,2.0,0,0,0,0,1,0
3,BR-ANVISA-300000008,2018-10-25,NaT,5 dia,Outro efeito clinicamente significativo,2025-11-17,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,12,11411,Sim,1,Recuperado/Resolvido,3,DIA,4,5.0,0,0,0,0,1,0
4,BR-ANVISA-300000010,NaT,NaT,None,Hospitalização/Prolongamento de hospitalização,2025-11-17,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,8,1994,Sim,1,Recuperado/Resolvido,3,DESCONHECIDO,0,NaN,0,0,0,1,0,0


### hash

In [33]:
hist_fat_reacoes.columns

Index(['IDENTIFICACAO_NOTIFICACAO', 'DATA_INICIO_HORA', 'DATA_FINAL_HORA',
       'DURACAO', 'GRAVIDADE', 'ATUALIZADO', 'HASH_BRONZE', 'SOC_CHAVE',
       'LLT_CHAVE', 'GRAVE_VALOR', 'GRAVE_CHAVE', 'DESFECHO_VALOR',
       'DESFECHO_CHAVE', 'DURACAO_TIPO_VALOR', 'DURACAO_TIPO_CHAVE',
       'DURACAO_VALOR', 'GRAVIDADE_RESULTADO_OBITO', 'GRAVIDADE_AMEACA_VIDA',
       'GRAVIDADE_INCAPACIDADE', 'GRAVIDADE_HOSPITALIZACAO',
       'GRAVIDADE_OUTRO_EFEITO', 'GRAVIDADE_ANOMALIA_CONGENITA'],
      dtype='object')

In [35]:
hist_fat_reacoes["HASH_SILVER"] = build_row_hash(
    hist_fat_reacoes, 
    hist_fat_reacoes.columns.difference(['ATUALIZADO', 'HASH_BRONZE']).tolist(), 
    algo="sha256", 
    sep="|"
)

In [35]:
hist_fat_reacoes.to_parquet(Path(project_root) / "data/02_silver/hist_fat_reacoes/hist_fat_reacoes.parquet", index=False)

# 🥇 Gold


In [39]:
gold_cols = ['IDENTIFICACAO_NOTIFICACAO','HASH_BRONZE', 'HASH_SILVER', 
        'DATA_INICIO_HORA', 
        'DATA_FINAL_HORA',
        'ATUALIZADO', 
       #'SOC_CHAVE',
       'LLT_CHAVE', 
       #'GRAVE_VALOR', 
       'GRAVE_CHAVE', 
       #'DESFECHO_VALOR',
       'DESFECHO_CHAVE', 
       'DURACAO_TIPO_CHAVE', 
       #'DURACAO_TIPO_VALOR',
       'DURACAO_VALOR', 
       'GRAVIDADE_RESULTADO_OBITO', 
       'GRAVIDADE_AMEACA_VIDA',
       'GRAVIDADE_INCAPACIDADE', 
       'GRAVIDADE_HOSPITALIZACAO',
       'GRAVIDADE_OUTRO_EFEITO', 
       'GRAVIDADE_ANOMALIA_CONGENITA']

In [40]:
fat_reacoes = hist_fat_reacoes[gold_cols].copy()
fat_reacoes.to_parquet(Path(project_root) / "data/03_gold/fat_reacoes/fat_reacoes.parquet", index=False)

In [41]:
fat_reacoes.head()

,IDENTIFICACAO_NOTIFICACAO,HASH_BRONZE,HASH_SILVER,DATA_INICIO_HORA,DATA_FINAL_HORA,ATUALIZADO,LLT_CHAVE,GRAVE_CHAVE,DESFECHO_CHAVE,DURACAO_TIPO_CHAVE,DURACAO_VALOR,GRAVIDADE_RESULTADO_OBITO,GRAVIDADE_AMEACA_VIDA,GRAVIDADE_INCAPACIDADE,GRAVIDADE_HOSPITALIZACAO,GRAVIDADE_OUTRO_EFEITO,GRAVIDADE_ANOMALIA_CONGENITA
0,BR-ANVISA-300000004,e0102770fa7fe4e3e3f58fe5a3675e13fffd3dc4721c0bdc390b6a0bb6bb8a59,5e664ee6fcb8b1bf52f1a7aa7ef1853a166bf9cedfc2ff2e86d54710236b01cf,NaT,NaT,2025-11-17,4114,2,3,4,3.0,0,0,0,0,0,0
1,BR-ANVISA-300000005,21a5c47062a6ba174af4058e70e681ff97ba91db76067626020d5936dfd44c38,497e70fa95ec2b543e9cb065388b857d545c25059f7f69cb902e8101cc8bcd0a,2018-11-22,2018-11-22,2025-11-17,9107,1,3,0,NaN,0,0,0,0,1,0
2,BR-ANVISA-300000007,de3dc592331e18cf642659152efae63272db0a105f20bd6e69410b1a28ae8823,b0af408ca8bd8d8cd9b2368aef19d38a8ec6787dfaa1408c31c7a80031bdba0d,2018-11-15,NaT,2025-11-17,3903,1,3,4,2.0,0,0,0,0,1,0
3,BR-ANVISA-300000008,9d8d2f4fb5c70d57589cdf2ca4a6c5d7b73f62cb62403ed4fff2761725e8e890,33dbb8ba6a8023f1281095e4a4f2ba8f6fbb6878c4410ea437b245d159c08ca0,2018-10-25,NaT,2025-11-17,11411,1,3,4,5.0,0,0,0,0,1,0
4,BR-ANVISA-300000010,57864a0b52f197b8c908c0ae1fa00bdedfe7c44adab7049d2152d2be4ea39eb2,10c2448dff13fcacd797f28c34696cadd5839a9a430953d12a7c4c6f14d4f772,NaT,NaT,2025-11-17,1994,1,3,0,NaN,0,0,0,1,0,0
